# Step 163 — Product Master Loading and Validation

## Objective
Load the synthetic oven product dataset and verify its structure before implementing the product recommendation engine.

## Dataset
The Product Master contains **500 oven configurations**, generated using predefined technical characteristics and configuration rules.

Each product is represented by a unique product identifier and its technical characteristics.

## Method
1. Load `product_master.csv` using Pandas.
2. Inspect the dataset dimensions and column names.
3. Display sample product configurations.
4. Check for missing values and duplicate records.
5. Examine the representation of multi-value characteristics such as `OVEN_FUNCTIONS`.

## Expected Outcome
A validated Product Master dataset ready for requirement-based product filtering and ranking.

## Next Step
Develop a product matching engine that:
- Applies **MUST** requirements as mandatory filters.
- Uses **PREFERRED** requirements to rank eligible products.
- Supports **EQUAL** and **CONTAINS** operators.
- Returns the most suitable product configurations.

## Project Architecture

**Customer Request → NLP Requirement Extraction → Priority Classification → Product Filtering → Product Ranking**

In [1]:
# Step 163 - Load and inspect Product Master

import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

# Load product master
product_path = PROCESSED_DIR / "product_master.csv"

assert product_path.exists(), (
    f"Product Master file not found: {product_path}"
)

product_master = pd.read_csv(product_path)

print("=== PRODUCT MASTER ===")
print("Products:", len(product_master))
print("Characteristics:", len(product_master.columns))

print("\n=== PRODUCT MASTER COLUMNS ===")
print(product_master.columns.tolist())

print("\n=== SAMPLE PRODUCTS ===")
display(product_master.head(5))

print("\n=== DATA QUALITY ===")
print("Missing values:", product_master.isna().sum().sum())
print("Duplicate rows:", product_master.duplicated().sum())

print("\nSTEP 163: COMPLETED")

=== PRODUCT MASTER ===
Products: 500
Characteristics: 25

=== PRODUCT MASTER COLUMNS ===
['PRODUCT_ID', 'WIDTH_CM', 'OVEN_FUNCTIONS', 'OVEN_VOLUME_L', 'HANDLE_TYPE', 'KNOB_TYPE', 'PANEL_COLOR', 'CLEANING', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'ENERGY_CLASS', 'PANEL_TYPE', 'WIFI', 'BAKING_TRAY', 'PACKAGING_TYPE', 'POWER_CABLE', 'ROASTING_TRAY', 'TELESCOPIC_RAILS', 'WIRE_SHELF']

=== SAMPLE PRODUCTS ===


,PRODUCT_ID,WIDTH_CM,OVEN_FUNCTIONS,OVEN_VOLUME_L,HANDLE_TYPE,KNOB_TYPE,PANEL_COLOR,CLEANING,HANDLE_COLOR,KNOB_COLOR,...,TURNSPIT,ENERGY_CLASS,PANEL_TYPE,WIFI,BAKING_TRAY,PACKAGING_TYPE,POWER_CABLE,ROASTING_TRAY,TELESCOPIC_RAILS,WIRE_SHELF
0,P0001,V003,V009,V002,V004,V004,V004,V004,V003,V003,...,V002,V001,V001,V001,V003,V001,V003,V002,V001,V003
1,P0002,V002,V011,V003,V004,V004,V003,V004,V003,V003,...,V002,V001,V001,V001,V002,V001,V001,V002,V003,V001
2,P0003,V004,V011,V004,V005,V005,V002,V004,V002,V001,...,V002,V002,V001,V001,V003,V001,V001,V003,V002,V002
3,P0004,V002,V006,V003,V004,V004,V004,V003,V001,V001,...,V002,V001,V001,V002,V002,V001,V004,V003,V003,V003
4,P0005,V001,V007,V001,V002,V005,V003,V003,V001,V003,...,V001,V001,V001,V002,V003,V001,V003,V001,V003,V002



=== DATA QUALITY ===
Missing values: 0
Duplicate rows: 0

STEP 163: COMPLETED


# Step 164 — Product Characteristic and Value Analysis

## Objective
Examine the technical value codes stored in the Product Master and determine how they can be matched with customer requirements extracted by the NLP model.

## Method
1. Identify all product characteristics.
2. Examine the unique values used for each characteristic.
3. Compare product characteristics with the Ground Truth structure.
4. Investigate how `OVEN_FUNCTIONS` combinations are represented.

## Expected Outcome
A clear mapping between NLP output labels and Product Master values.

This mapping will support accurate product filtering using mandatory requirements and ranking using optional preferences.

In [2]:
# Step 164 - Analyze Product Master characteristics and values

import pandas as pd

characteristic_columns = [
    col for col in product_master.columns
    if col != "PRODUCT_ID"
]

# Count unique values for each characteristic
characteristic_summary = pd.DataFrame([
    {
        "CHARACTERISTIC": col,
        "UNIQUE_VALUES": product_master[col].nunique(),
        "SAMPLE_VALUES": ", ".join(
            product_master[col].astype(str).unique()[:10]
        )
    }
    for col in characteristic_columns
])

print("=== PRODUCT CHARACTERISTIC SUMMARY ===")
display(characteristic_summary)

# Inspect oven function representation
print("\n=== OVEN FUNCTIONS IN PRODUCT MASTER ===")

oven_function_values = (
    product_master["OVEN_FUNCTIONS"]
    .value_counts()
    .rename_axis("VALUE")
    .reset_index(name="PRODUCT_COUNT")
)

display(oven_function_values)

# Check Ground Truth function values
ground_truth_path = PROCESSED_DIR / "nlp_ground_truth.csv"

ground_truth_matching = pd.read_csv(ground_truth_path)

print("\n=== OVEN FUNCTIONS IN GROUND TRUTH ===")

gt_oven_functions = (
    ground_truth_matching[
        ground_truth_matching["CHARACTERISTIC"] == "OVEN_FUNCTIONS"
    ][["VALUE", "VALUE_DEFINITION", "OPERATOR"]]
    .drop_duplicates()
    .sort_values("VALUE")
)

display(gt_oven_functions)

print("\nSTEP 164: COMPLETED")

=== PRODUCT CHARACTERISTIC SUMMARY ===


,CHARACTERISTIC,UNIQUE_VALUES,SAMPLE_VALUES
0,WIDTH_CM,4,"V003, V002, V004, V001"
1,OVEN_FUNCTIONS,12,"V009, V011, V006, V007, V005, V010, V003, V002..."
2,OVEN_VOLUME_L,4,"V002, V003, V004, V001"
3,HANDLE_TYPE,5,"V004, V005, V002, V001, V003"
4,KNOB_TYPE,5,"V004, V005, V002, V001, V003"
5,PANEL_COLOR,5,"V004, V003, V002, V001, V005"
6,CLEANING,4,"V004, V003, V002, V001"
7,HANDLE_COLOR,4,"V003, V002, V001, V004"
8,KNOB_COLOR,4,"V003, V001, V004, V002"
9,KNOB_STRUCTURE,2,"V002, V001"



=== OVEN FUNCTIONS IN PRODUCT MASTER ===


,VALUE,PRODUCT_COUNT
0,V005,63
1,V006,58
2,V007,48
3,V009,46
4,V004,46
5,V003,41
6,V010,39
7,V008,38
8,V002,33
9,V012,33



=== OVEN FUNCTIONS IN GROUND TRUTH ===


,VALUE,VALUE_DEFINITION,OPERATOR
2770,3D,3D,CONTAINS
2822,Bottom,Bottom,CONTAINS
2762,Bottom+Fan,Bottom+Fan,CONTAINS
2764,Defrost,Defrost,CONTAINS
2773,DoubleGrill,DoubleGrill,CONTAINS
2771,DoubleGrill+Fan,DoubleGrill+Fan,CONTAINS
2765,DoubleGrill+Fan+Turnspit,DoubleGrill+Fan+Turnspit,CONTAINS
2799,DoubleGrill+Turnspit,DoubleGrill+Turnspit,CONTAINS
2766,Fan+Ring,Fan+Ring,CONTAINS
2774,Grill,Grill,CONTAINS



STEP 164: COMPLETED


In [3]:
# Step 165 - Load and normalize oven function combinations

import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/oven_function_set.csv")

assert file_path.exists(), f"File not found: {file_path}"

# The source table has no header row
function_set = pd.read_csv(
    file_path,
    header=None,
    dtype=str,
    encoding="utf-8-sig"
)

# Column B contains combination codes
# Columns E onward contain individual cooking functions

function_mapping = {}

for _, row in function_set.iterrows():

    combination_code = str(row.iloc[1]).strip()

    functions = [
        str(value).strip()
        for value in row.iloc[4:]
        if pd.notna(value) and str(value).strip()
    ]

    function_mapping[combination_code] = functions

print("=== OVEN FUNCTION MAPPING ===")
print("Combinations:", len(function_mapping))

for code, functions in function_mapping.items():
    print(f"{code}: {functions}")

# Verify coverage against Product Master
product_codes = set(
    product_master["OVEN_FUNCTIONS"].astype(str)
)

mapping_codes = set(function_mapping)

missing_codes = product_codes - mapping_codes

print("\n=== MAPPING VALIDATION ===")
print("Product codes:", len(product_codes))
print("Mapped codes:", len(mapping_codes))
print("Missing codes:", missing_codes)

assert not missing_codes, (
    f"Unmapped oven function codes: {missing_codes}"
)

print("\nSTEP 165: COMPLETED")

=== OVEN FUNCTION MAPPING ===
Combinations: 13
nan: []
V001: ['Top+Bottom', 'Bottom', 'Top']
V002: ['Defrost', 'Top+Bottom', 'Top+Bottom+Fan', 'Grill', 'Grill+Fan']
V003: ['Bottom', 'Top', 'Top+Bottom', 'Top+Bottom+Fan', 'Bottom+Fan']
V004: ['OvenLight', 'Defrost', 'Top+Bottom', 'Top+Bottom+Fan', 'Bottom+Fan', 'Grill+Turnspit', 'DoubleGrill+Turnspit', 'DoubleGrill+Fan+Turnspit']
V005: ['OvenLight', 'Defrost', 'Top+Bottom', 'Top+Bottom+Fan', 'Bottom+Fan', 'Grill', 'DoubleGrill', 'DoubleGrill+Fan']
V006: ['Defrost', 'Fan+Ring', 'Top+Bottom', 'Top+Bottom+Fan', 'Grill', 'DoubleGrill', 'DoubleGrill+Fan', 'Pyrolysis']
V007: ['Defrost', 'Fan+Ring', 'Top+Bottom', 'Top+Bottom+Fan', 'Grill+Turnspit', 'DoubleGrill+Turnspit', 'DoubleGrill+Fan+Turnspit', 'Pyrolysis']
V008: ['OvenLight', 'Defrost', 'Fan+Ring', 'Top+Bottom', 'Top+Bottom+Fan', 'Pizza', 'Grill', 'DoubleGrill', 'DoubleGrill+Fan', 'HydroClean']
V009: ['Defrost', 'TURBO', 'Bottom', 'Top+Bottom', 'Top+Bottom+Fan', 'Pizza', 'DoubleGrill', '

## STEP 166 — Validate and Clean Oven Function Mapping

This step validates the oven function mapping created in Step 165.

Objectives:
- Remove invalid or missing mapping keys.
- Verify that all oven function codes used in Product Master are mapped.
- Check that each mapped code contains at least one oven function.
- Prepare a clean mapping dictionary for product matching.

In [4]:
# STEP 166 — Validate and Clean Oven Function Mapping

import pandas as pd

# Remove invalid mapping keys and normalize codes
function_mapping_clean = {
    str(code).strip(): functions
    for code, functions in function_mapping.items()
    if pd.notna(code)
    and str(code).strip()
    and str(code).strip().lower() != "nan"
}

# Get oven function codes used in Product Master
product_function_codes = set(
    product_master["OVEN_FUNCTIONS"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Get mapped codes
mapped_function_codes = set(function_mapping_clean.keys())

# Validation checks
missing_codes = product_function_codes - mapped_function_codes

empty_mappings = {
    code: functions
    for code, functions in function_mapping_clean.items()
    if not functions
}

print("=== OVEN FUNCTION MAPPING VALIDATION ===")
print(f"Product codes: {len(product_function_codes)}")
print(f"Mapped codes: {len(mapped_function_codes)}")
print(f"Missing codes: {missing_codes}")
print(f"Empty mappings: {empty_mappings}")

assert not missing_codes, "Some product function codes are missing!"
assert not empty_mappings, "Some function mappings are empty!"

print("\nSTEP 166: COMPLETED")

=== OVEN FUNCTION MAPPING VALIDATION ===
Product codes: 12
Mapped codes: 12
Missing codes: set()
Empty mappings: {}

STEP 166: COMPLETED


## STEP 167 — Define Product Requirement Matching Logic

This step implements the core matching logic between customer requirements
and product characteristics.

Two operators are supported:

- **EQUAL:** Checks whether the product characteristic value equals the requested value.
- **CONTAINS:** Checks whether an oven function combination includes the requested function.

The matching function returns:
- `True` if the requirement is satisfied.
- `False` if the requirement is not satisfied.

This function will be used to evaluate individual customer requirements
against the synthetic product portfolio.

In [5]:
# STEP 167 — Define Product Requirement Matching Logic

import pandas as pd


def match_requirement(product, requirement):
    """
    Check whether a product satisfies a customer requirement.

    Parameters
    ----------
    product : pandas.Series
        One row from Product Master.

    requirement : pandas.Series
        One row from Ground Truth.

    Returns
    -------
    bool
        True if the requirement is satisfied, otherwise False.
    """

    characteristic = str(requirement["CHARACTERISTIC"]).strip()
    operator = str(requirement["OPERATOR"]).strip().upper()
    required_value = str(requirement["VALUE"]).strip()

    # Check whether the characteristic exists in Product Master
    if characteristic not in product.index:
        return False

    product_value = product[characteristic]

    if pd.isna(product_value):
        return False

    product_value = str(product_value).strip()

    # EQUAL: direct characteristic-value comparison
    if operator == "EQUAL":
        return product_value == required_value

    # CONTAINS: oven function membership
    elif operator == "CONTAINS":

        if characteristic != "OVEN_FUNCTIONS":
            return False

        available_functions = function_mapping_clean.get(
            product_value, []
        )

        return required_value in available_functions

    # Unsupported operator
    return False


print("STEP 167: MATCHING FUNCTION DEFINED")

STEP 167: MATCHING FUNCTION DEFINED


## STEP 168 — Test Product Requirement Matching

Validate the matching function using actual Product Master and Ground Truth records.

Objectives:
- Test EQUAL matching for characteristic-value codes.
- Test CONTAINS matching for oven functions.
- Verify that matching and non-matching cases return the expected results.
- Identify potential data format inconsistencies before evaluating the full product portfolio.

In [6]:
# STEP 168 — Load Ground Truth and Test Matching

import pandas as pd

# Load validated NLP Ground Truth
ground_truth_path = PROCESSED_DIR / "nlp_ground_truth.csv"

assert ground_truth_path.exists(), (
    f"Ground Truth file not found: {ground_truth_path}"
)

ground_truth = pd.read_csv(ground_truth_path)

print("=== GROUND TRUTH ===")
print("Requirements:", len(ground_truth))

print("\n=== GROUND TRUTH COLUMNS ===")
print(ground_truth.columns.tolist())

print("\n=== SAMPLE REQUIREMENTS ===")
display(ground_truth.head(5))

print("\n=== OPERATOR DISTRIBUTION ===")
print(ground_truth["OPERATOR"].value_counts())

print("\nSTEP 168: GROUND TRUTH LOADED")

=== GROUND TRUTH ===
Requirements: 7727

=== GROUND TRUTH COLUMNS ===
['REQUEST_ID', 'CHARACTERISTIC', 'VALUE', 'VALUE_DEFINITION', 'OPERATOR', 'REQUIREMENT_TYPE']

=== SAMPLE REQUIREMENTS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,VALUE_DEFINITION,OPERATOR,REQUIREMENT_TYPE
0,R0001,OVEN_TYPE,V002,STEAM,EQUAL,MUST
1,R0001,WIDTH_CM,V001,45,EQUAL,MUST
2,R0002,OVEN_VOLUME_L,V001,50,EQUAL,MUST
3,R0002,PACKAGING_TYPE,V001,SHRINK,EQUAL,MUST
4,R0002,PANEL_COLOR,V002,WHITE,EQUAL,MUST



=== OPERATOR DISTRIBUTION ===
OPERATOR
EQUAL       5201
CONTAINS    2526
Name: count, dtype: int64

STEP 168: GROUND TRUTH LOADED


## STEP 169 — Validate Requirement Matching

This step validates the requirement matching function using controlled test cases.

Objectives:
- Verify positive and negative EQUAL matching.
- Verify positive and negative CONTAINS matching.
- Confirm that oven function combination codes are correctly interpreted.
- Identify any inconsistencies before full product matching.

In [7]:
# STEP 169 — Validate Requirement Matching

import pandas as pd

sample_product = product_master.iloc[0].copy()

# Test 1: EQUAL — Matching value
equal_characteristic = "WIDTH_CM"
correct_value = str(sample_product[equal_characteristic]).strip()

equal_positive = pd.Series({
    "CHARACTERISTIC": equal_characteristic,
    "VALUE": correct_value,
    "OPERATOR": "EQUAL"
})

# Test 2: EQUAL — Non-matching value
equal_negative = pd.Series({
    "CHARACTERISTIC": equal_characteristic,
    "VALUE": "__INVALID_VALUE__",
    "OPERATOR": "EQUAL"
})

# Test 3: CONTAINS — Matching function
combination_code = str(sample_product["OVEN_FUNCTIONS"]).strip()

available_functions = function_mapping_clean[combination_code]

assert available_functions, (
    f"No functions found for combination {combination_code}"
)

contains_positive = pd.Series({
    "CHARACTERISTIC": "OVEN_FUNCTIONS",
    "VALUE": available_functions[0],
    "OPERATOR": "CONTAINS"
})

# Test 4: CONTAINS — Non-matching function
contains_negative = pd.Series({
    "CHARACTERISTIC": "OVEN_FUNCTIONS",
    "VALUE": "__INVALID_FUNCTION__",
    "OPERATOR": "CONTAINS"
})

# Execute tests
test_cases = [
    ("EQUAL - Positive", equal_positive, True),
    ("EQUAL - Negative", equal_negative, False),
    ("CONTAINS - Positive", contains_positive, True),
    ("CONTAINS - Negative", contains_negative, False)
]

results = []

for test_name, requirement, expected in test_cases:

    actual = match_requirement(sample_product, requirement)

    results.append({
        "TEST": test_name,
        "EXPECTED": expected,
        "ACTUAL": actual,
        "PASS": actual == expected
    })

results_df = pd.DataFrame(results)

print("=== MATCHING FUNCTION TEST RESULTS ===")
display(results_df)

assert results_df["PASS"].all(), (
    "Some matching tests failed!"
)

print("\nSTEP 169: COMPLETED")

=== MATCHING FUNCTION TEST RESULTS ===


,TEST,EXPECTED,ACTUAL,PASS
0,EQUAL - Positive,True,True,True
1,EQUAL - Negative,False,False,True
2,CONTAINS - Positive,True,True,True
3,CONTAINS - Negative,False,False,True



STEP 169: COMPLETED


## STEP 170 — Validate Ground Truth and Product Value Compatibility

This step checks the compatibility between Ground Truth requirements
and Product Master characteristics.

Validation objectives:
- Identify characteristics missing from Product Master.
- Verify EQUAL requirement values against available product values.
- Verify CONTAINS requirement values against normalized oven functions.
- Detect incompatible values before full product matching.

This validation checks whether requirement values are represented
in the product portfolio, not whether every customer request can
be fully satisfied.

In [8]:
# STEP 170 — Validate Ground Truth and Product Value Compatibility

import pandas as pd

# Work with a copy
gt_check = ground_truth.copy()

# Normalize relevant fields
for col in ["CHARACTERISTIC", "VALUE", "OPERATOR"]:
    gt_check[col] = gt_check[col].astype(str).str.strip()

gt_check["OPERATOR"] = gt_check["OPERATOR"].str.upper()

# 1. Check characteristics
product_characteristics = set(product_master.columns)

missing_characteristics = (
    set(gt_check["CHARACTERISTIC"]) - product_characteristics
)

# 2. Validate EQUAL requirements
equal_gt = gt_check[gt_check["OPERATOR"] == "EQUAL"].copy()

equal_results = []

for characteristic, group in equal_gt.groupby("CHARACTERISTIC"):

    if characteristic not in product_master.columns:
        continue

    product_values = set(
        product_master[characteristic]
        .dropna()
        .astype(str)
        .str.strip()
    )

    requested_values = set(group["VALUE"])

    missing_values = requested_values - product_values

    equal_results.append({
        "CHARACTERISTIC": characteristic,
        "REQUESTED_VALUES": len(requested_values),
        "PRODUCT_VALUES": len(product_values),
        "MISSING_VALUES": sorted(missing_values)
    })

equal_validation = pd.DataFrame(equal_results)

# 3. Validate CONTAINS requirements
contains_gt = gt_check[
    gt_check["OPERATOR"] == "CONTAINS"
].copy()

available_functions = {
    function
    for functions in function_mapping_clean.values()
    for function in functions
}

requested_functions = set(contains_gt["VALUE"])

missing_functions = requested_functions - available_functions

# 4. Display results
print("=== GROUND TRUTH / PRODUCT COMPATIBILITY ===")

print("\nMissing characteristics:")
print(missing_characteristics)

print("\n=== EQUAL VALUE VALIDATION ===")
display(equal_validation)

print("\n=== CONTAINS VALUE VALIDATION ===")
print("Requested functions:", len(requested_functions))
print("Available functions:", len(available_functions))
print("Missing functions:", sorted(missing_functions))

print("\nSTEP 170: COMPLETED")

=== GROUND TRUTH / PRODUCT COMPATIBILITY ===

Missing characteristics:
set()

=== EQUAL VALUE VALIDATION ===


,CHARACTERISTIC,REQUESTED_VALUES,PRODUCT_VALUES,MISSING_VALUES
0,BAKING_TRAY,3,3,[]
1,CLEANING,4,4,[]
2,ENERGY_CLASS,3,3,[]
3,HANDLE_COLOR,4,4,[]
4,HANDLE_TYPE,5,5,[]
5,KNOB_COLOR,4,4,[]
6,KNOB_STRUCTURE,2,2,[]
7,KNOB_TYPE,5,5,[]
8,NUMBER_OF_FUNCTIONS,5,5,[]
9,OVEN_CONTROL,3,3,[]



=== CONTAINS VALUE VALIDATION ===
Requested functions: 21
Available functions: 21
Missing functions: []

STEP 170: COMPLETED


## STEP 171 — Match Products Against MUST Requirements

This step implements the first stage of product matching.

For a selected customer request:
- Retrieve all MUST requirements from Ground Truth.
- Evaluate each product against these requirements.
- Retain products that satisfy every MUST requirement.
- Report the number of eligible products.

A product is eligible only if all mandatory requirements are satisfied.

PREFERRED requirements are not considered at this stage.

In [9]:
# STEP 171 — Match Products Against MUST Requirements

import pandas as pd

# Inspect requirement priority distribution
print("=== REQUIREMENT PRIORITY DISTRIBUTION ===")
print(ground_truth["REQUIREMENT_TYPE"].value_counts())

# Select the first customer request
sample_request_id = ground_truth["REQUEST_ID"].iloc[0]

# Retrieve requirements for the selected request
request_requirements = ground_truth[
    ground_truth["REQUEST_ID"] == sample_request_id
].copy()

# Select MUST requirements
must_requirements = request_requirements[
    request_requirements["REQUIREMENT_TYPE"] == "MUST"
]

print("\n=== SELECTED CUSTOMER REQUEST ===")
print("Request ID:", sample_request_id)
print("Total requirements:", len(request_requirements))
print("MUST requirements:", len(must_requirements))

display(
    must_requirements[
        ["CHARACTERISTIC", "VALUE", "OPERATOR", "REQUIREMENT_TYPE"]
    ]
)

# Evaluate each product against all MUST requirements
eligible_product_indices = []

for product_index, product in product_master.iterrows():

    all_must_satisfied = all(
        match_requirement(product, requirement)
        for _, requirement in must_requirements.iterrows()
    )

    if all_must_satisfied:
        eligible_product_indices.append(product_index)

# Create eligible product subset
eligible_products = product_master.loc[
    eligible_product_indices
].copy()

print("\n=== MUST MATCHING RESULTS ===")
print("Total products:", len(product_master))
print("Eligible products:", len(eligible_products))
print(
    "Eliminated products:",
    len(product_master) - len(eligible_products)
)

print("\n=== SAMPLE ELIGIBLE PRODUCTS ===")
display(eligible_products.head(10))

print("\nSTEP 171: COMPLETED")

=== REQUIREMENT PRIORITY DISTRIBUTION ===
REQUIREMENT_TYPE
MUST         5834
PREFERRED    1893
Name: count, dtype: int64

=== SELECTED CUSTOMER REQUEST ===
Request ID: R0001
Total requirements: 2
MUST requirements: 2


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,OVEN_TYPE,V002,EQUAL,MUST
1,WIDTH_CM,V001,EQUAL,MUST



=== MUST MATCHING RESULTS ===
Total products: 500
Eligible products: 3
Eliminated products: 497

=== SAMPLE ELIGIBLE PRODUCTS ===


,PRODUCT_ID,WIDTH_CM,OVEN_FUNCTIONS,OVEN_VOLUME_L,HANDLE_TYPE,KNOB_TYPE,PANEL_COLOR,CLEANING,HANDLE_COLOR,KNOB_COLOR,...,TURNSPIT,ENERGY_CLASS,PANEL_TYPE,WIFI,BAKING_TRAY,PACKAGING_TYPE,POWER_CABLE,ROASTING_TRAY,TELESCOPIC_RAILS,WIRE_SHELF
74,P0075,V001,V010,V001,V003,V002,V003,V004,V001,V003,...,V002,V003,V001,V001,V001,V002,V005,V001,V002,V003
139,P0140,V001,V010,V001,V003,V003,V004,V004,V001,V001,...,V002,V003,V001,V001,V001,V001,V001,V003,V001,V003
448,P0449,V001,V010,V001,V004,V002,V002,V004,V001,V001,...,V002,V003,V001,V001,V001,V002,V002,V003,V003,V003



STEP 171: COMPLETED


## STEP 172 — Rank Eligible Products by PREFERRED Requirements

This step implements the second stage of product matching.

Matching strategy:
- MUST requirements are mandatory eligibility constraints.
- PREFERRED requirements are used to rank eligible products.
- Each satisfied PREFERRED requirement contributes one point.
- Products are ranked by their total preference score.

All PREFERRED requirements have equal weight in this baseline approach.

The matching process uses Ground Truth requirements to establish
a reference product-ranking mechanism.

In [10]:
# STEP 172 — Rank Eligible Products by PREFERRED Requirements

import pandas as pd


def rank_products_for_request(request_id):

    # Retrieve requirements
    request_reqs = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    must_reqs = request_reqs[
        request_reqs["REQUIREMENT_TYPE"] == "MUST"
    ]

    preferred_reqs = request_reqs[
        request_reqs["REQUIREMENT_TYPE"] == "PREFERRED"
    ]

    results = []

    for _, product in product_master.iterrows():

        # Stage 1: MUST filtering
        must_satisfied = all(
            match_requirement(product, req)
            for _, req in must_reqs.iterrows()
        )

        if not must_satisfied:
            continue

        # Stage 2: PREFERRED scoring
        preferred_score = sum(
            match_requirement(product, req)
            for _, req in preferred_reqs.iterrows()
        )

        results.append({
            "REQUEST_ID": request_id,
            "PRODUCT_ID": product["PRODUCT_ID"],
            "MUST_COUNT": len(must_reqs),
            "PREFERRED_COUNT": len(preferred_reqs),
            "PREFERRED_SCORE": preferred_score
        })

    ranked_products = pd.DataFrame(
        results,
        columns=[
            "REQUEST_ID",
            "PRODUCT_ID",
            "MUST_COUNT",
            "PREFERRED_COUNT",
            "PREFERRED_SCORE"
        ]
    )

    if not ranked_products.empty:
        ranked_products = ranked_products.sort_values(
            by=["PREFERRED_SCORE", "PRODUCT_ID"],
            ascending=[False, True]
        ).reset_index(drop=True)

    return ranked_products


# Test 1: R0001
print("=== REQUEST R0001 ===")

ranked_r0001 = rank_products_for_request("R0001")

display(ranked_r0001.head(10))


# Test 2: Find a request with PREFERRED requirements
preferred_request_ids = ground_truth.loc[
    ground_truth["REQUIREMENT_TYPE"] == "PREFERRED",
    "REQUEST_ID"
].unique()

sample_preferred_request_id = preferred_request_ids[0]

print("\n=== REQUEST WITH PREFERRED REQUIREMENTS ===")
print("Request ID:", sample_preferred_request_id)

ranked_preferred = rank_products_for_request(
    sample_preferred_request_id
)

print("Eligible products:", len(ranked_preferred))

display(ranked_preferred.head(10))

print("\nSTEP 172: COMPLETED")

=== REQUEST R0001 ===


,REQUEST_ID,PRODUCT_ID,MUST_COUNT,PREFERRED_COUNT,PREFERRED_SCORE
0,R0001,P0075,2,0,0
1,R0001,P0140,2,0,0
2,R0001,P0449,2,0,0



=== REQUEST WITH PREFERRED REQUIREMENTS ===
Request ID: R0003
Eligible products: 0


,REQUEST_ID,PRODUCT_ID,MUST_COUNT,PREFERRED_COUNT,PREFERRED_SCORE



STEP 172: COMPLETED


## STEP 173 — Find a Valid Request for Preference Ranking

The initial preference-ranking test returned no eligible products.

This step searches for a customer request that:
- Contains at least one PREFERRED requirement.
- Has at least one product satisfying all MUST requirements.

The objective is to validate preference scoring using a real
customer request with eligible products.

In [11]:
# STEP 173 — Find a Valid Request for Preference Ranking

import pandas as pd

# Requests containing PREFERRED requirements
preferred_request_ids = ground_truth.loc[
    ground_truth["REQUIREMENT_TYPE"] == "PREFERRED",
    "REQUEST_ID"
].unique()

selected_request_id = None
selected_ranking = None

for request_id in preferred_request_ids:

    ranking = rank_products_for_request(request_id)

    if not ranking.empty:
        selected_request_id = request_id
        selected_ranking = ranking
        break

print("=== PREFERENCE RANKING TEST ===")

if selected_request_id is not None:

    print("Selected Request ID:", selected_request_id)
    print("Eligible products:", len(selected_ranking))

    print("\n=== CUSTOMER REQUIREMENTS ===")

    selected_requirements = ground_truth[
        ground_truth["REQUEST_ID"] == selected_request_id
    ]

    display(
        selected_requirements[
            [
                "CHARACTERISTIC",
                "VALUE",
                "OPERATOR",
                "REQUIREMENT_TYPE"
            ]
        ]
    )

    print("\n=== RANKED PRODUCTS ===")
    display(selected_ranking.head(10))

    print("\n=== PREFERENCE SCORE DISTRIBUTION ===")
    print(
        selected_ranking["PREFERRED_SCORE"]
        .value_counts()
        .sort_index(ascending=False)
    )

else:
    print("No request with PREFERRED requirements and eligible products found.")

print("\nSTEP 173: COMPLETED")

=== PREFERENCE RANKING TEST ===
Selected Request ID: R0004
Eligible products: 38

=== CUSTOMER REQUIREMENTS ===


,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
11,HANDLE_COLOR,V001,EQUAL,PREFERRED
12,HANDLE_TYPE,V002,EQUAL,MUST
13,NUMBER_OF_FUNCTIONS,V003,EQUAL,MUST



=== RANKED PRODUCTS ===


,REQUEST_ID,PRODUCT_ID,MUST_COUNT,PREFERRED_COUNT,PREFERRED_SCORE
0,R0004,P0005,2,1,1
1,R0004,P0059,2,1,1
2,R0004,P0084,2,1,1
3,R0004,P0094,2,1,1
4,R0004,P0111,2,1,1
5,R0004,P0122,2,1,1
6,R0004,P0206,2,1,1
7,R0004,P0251,2,1,1
8,R0004,P0277,2,1,1
9,R0004,P0320,2,1,1



=== PREFERENCE SCORE DISTRIBUTION ===
PREFERRED_SCORE
1    15
0    23
Name: count, dtype: int64

STEP 173: COMPLETED


## STEP 174 — Evaluate Product Matching Across All Customer Requests

This step evaluates the complete validated customer request dataset
using Ground Truth requirements.

Objectives:
- Identify requests with at least one eligible product.
- Identify requests with no products satisfying all MUST requirements.
- Calculate the number of eligible products per request.
- Evaluate PREFERRED requirement satisfaction.
- Establish baseline product matching performance.

This evaluation uses Ground Truth rather than NLP predictions.

In [12]:
# STEP 174 — Evaluate Product Matching Across All Customer Requests

import pandas as pd

request_ids = ground_truth["REQUEST_ID"].unique()

matching_summary = []

for request_id in request_ids:

    # Retrieve requirements
    request_reqs = ground_truth[
        ground_truth["REQUEST_ID"] == request_id
    ]

    must_count = (
        request_reqs["REQUIREMENT_TYPE"] == "MUST"
    ).sum()

    preferred_count = (
        request_reqs["REQUIREMENT_TYPE"] == "PREFERRED"
    ).sum()

    # Match and rank products
    ranked = rank_products_for_request(request_id)

    eligible_count = len(ranked)

    if eligible_count > 0:
        max_preferred_score = int(
            ranked["PREFERRED_SCORE"].max()
        )
    else:
        max_preferred_score = 0

    # Full preference satisfaction requires an eligible product
    all_preferences_satisfied = (
        eligible_count > 0
        and max_preferred_score == preferred_count
    )

    matching_summary.append({
        "REQUEST_ID": request_id,
        "MUST_COUNT": must_count,
        "PREFERRED_COUNT": preferred_count,
        "ELIGIBLE_PRODUCTS": eligible_count,
        "HAS_MATCH": eligible_count > 0,
        "MAX_PREFERRED_SCORE": max_preferred_score,
        "ALL_PREFERENCES_SATISFIED": all_preferences_satisfied
    })

matching_summary = pd.DataFrame(matching_summary)

# Summary metrics
total_requests = len(matching_summary)

matched_requests = int(
    matching_summary["HAS_MATCH"].sum()
)

unmatched_requests = total_requests - matched_requests

match_rate = matched_requests / total_requests

print("=== PRODUCT MATCHING BASELINE ===")

print("Total requests:", total_requests)
print("Requests with eligible products:", matched_requests)
print("Requests without eligible products:", unmatched_requests)
print(f"Request match rate: {match_rate:.2%}")

print("\n=== ELIGIBLE PRODUCT DISTRIBUTION ===")

print(
    matching_summary["ELIGIBLE_PRODUCTS"].describe()
)

print("\n=== PREFERENCE SATISFACTION ===")

print(
    "Requests with all preferences satisfied:",
    matching_summary["ALL_PREFERENCES_SATISFIED"].sum()
)

print("\n=== SAMPLE MATCHING RESULTS ===")

display(matching_summary.head(10))

print("\nSTEP 174: COMPLETED")

=== PRODUCT MATCHING BASELINE ===
Total requests: 1967
Requests with eligible products: 1750
Requests without eligible products: 217
Request match rate: 88.97%

=== ELIGIBLE PRODUCT DISTRIBUTION ===
count    1967.000000
mean       66.043721
std        92.186137
min         0.000000
25%         8.000000
50%        32.000000
75%        88.500000
max       500.000000
Name: ELIGIBLE_PRODUCTS, dtype: float64

=== PREFERENCE SATISFACTION ===
Requests with all preferences satisfied: 1654

=== SAMPLE MATCHING RESULTS ===


,REQUEST_ID,MUST_COUNT,PREFERRED_COUNT,ELIGIBLE_PRODUCTS,HAS_MATCH,MAX_PREFERRED_SCORE,ALL_PREFERENCES_SATISFIED
0,R0001,2,0,3,True,0,True
1,R0002,3,0,4,True,0,True
2,R0003,5,1,0,False,0,False
3,R0004,2,1,38,True,1,True
4,R0005,4,1,2,True,1,True
5,R0007,3,2,8,True,2,True
6,R0008,6,0,0,False,0,False
7,R0009,3,1,3,True,1,True
8,R0012,2,0,50,True,0,True
9,R0014,5,1,0,False,0,False



STEP 174: COMPLETED


## STEP 175 — Analyze Matching Coverage and Preference Satisfaction

This step provides a detailed analysis of the Ground Truth
product matching baseline.

Objectives:
- Identify requests without mandatory requirements.
- Evaluate preference satisfaction only for requests containing
  PREFERRED requirements.
- Analyze requests without eligible products.
- Investigate how the number of MUST requirements affects
  product availability.

The analysis distinguishes product portfolio coverage from
NLP prediction performance.

In [13]:
# STEP 175 — Analyze Matching Coverage and Preference Satisfaction

import pandas as pd

# 1. Requests without MUST requirements
no_must_requests = matching_summary[
    matching_summary["MUST_COUNT"] == 0
]

print("=== REQUESTS WITHOUT MUST REQUIREMENTS ===")
print("Requests:", len(no_must_requests))


# 2. Preference satisfaction analysis
preferred_requests = matching_summary[
    matching_summary["PREFERRED_COUNT"] > 0
]

preferred_with_match = preferred_requests[
    preferred_requests["HAS_MATCH"]
]

preferred_fully_satisfied = preferred_with_match[
    preferred_with_match["ALL_PREFERENCES_SATISFIED"]
]

print("\n=== PREFERENCE SATISFACTION ANALYSIS ===")

print("Requests with PREFERRED requirements:",
      len(preferred_requests))

print("Requests with PREFERRED and eligible products:",
      len(preferred_with_match))

print("Requests with all preferences satisfied:",
      len(preferred_fully_satisfied))

if len(preferred_with_match) > 0:

    preference_satisfaction_rate = (
        len(preferred_fully_satisfied)
        / len(preferred_with_match)
    )

    print(
        f"Full preference satisfaction rate: "
        f"{preference_satisfaction_rate:.2%}"
    )


# 3. Requests without eligible products
unmatched_requests = matching_summary[
    ~matching_summary["HAS_MATCH"]
]

print("\n=== UNMATCHED REQUEST ANALYSIS ===")

print("Unmatched requests:", len(unmatched_requests))

print("\nMUST requirement count distribution:")

print(
    unmatched_requests["MUST_COUNT"]
    .value_counts()
    .sort_index()
)


# 4. Matching rate by MUST requirement count
coverage_by_must = (
    matching_summary
    .groupby("MUST_COUNT")
    .agg(
        REQUEST_COUNT=("REQUEST_ID", "count"),
        MATCHED_REQUESTS=("HAS_MATCH", "sum"),
        AVG_ELIGIBLE_PRODUCTS=("ELIGIBLE_PRODUCTS", "mean")
    )
    .reset_index()
)

coverage_by_must["MATCH_RATE"] = (
    coverage_by_must["MATCHED_REQUESTS"]
    / coverage_by_must["REQUEST_COUNT"]
)

print("\n=== MATCHING COVERAGE BY MUST COUNT ===")

display(coverage_by_must)

print("\nSTEP 175: COMPLETED")

=== REQUESTS WITHOUT MUST REQUIREMENTS ===
Requests: 1

=== PREFERENCE SATISFACTION ANALYSIS ===
Requests with PREFERRED requirements: 1300
Requests with PREFERRED and eligible products: 1201
Requests with all preferences satisfied: 1105
Full preference satisfaction rate: 92.01%

=== UNMATCHED REQUEST ANALYSIS ===
Unmatched requests: 217

MUST requirement count distribution:
MUST_COUNT
2    16
3    45
4    67
5    62
6    27
Name: count, dtype: int64

=== MATCHING COVERAGE BY MUST COUNT ===


,MUST_COUNT,REQUEST_COUNT,MATCHED_REQUESTS,AVG_ELIGIBLE_PRODUCTS,MATCH_RATE
0,0,1,1,500.000000,1.000000
1,1,227,227,213.577093,1.000000
2,2,516,500,86.213178,0.968992
3,3,593,548,43.998314,0.924115
4,4,401,334,20.942643,0.832918
5,5,182,120,9.593407,0.659341
6,6,47,20,4.361702,0.425532



STEP 175: COMPLETED


## STEP 176 — Independent Product Matching Validation

This step independently validates the Ground Truth product matching results.

Validation objectives:
- Recalculate product eligibility without using the existing matching functions.
- Compare eligible Product IDs for every customer request.
- Independently recalculate PREFERRED scores.
- Identify false inclusions, false exclusions, and scoring mismatches.
- Verify the consistency of the matching baseline.

The validation uses Ground Truth requirements and Product Master data.

This is an implementation consistency check, not an evaluation
of NLP prediction accuracy.

In [14]:
# STEP 176 — Independent Product Matching Validation

import pandas as pd

# --------------------------------------------------
# 1. Prepare independent product representation
# --------------------------------------------------

products_check = product_master.copy()

products_check["PRODUCT_ID"] = (
    products_check["PRODUCT_ID"].astype(str).str.strip()
)

# Normalize product characteristic values
for col in products_check.columns:
    if col != "PRODUCT_ID":
        products_check[col] = (
            products_check[col].astype("string").str.strip()
        )

# Independently expand oven function combinations
product_function_sets = {
    product_id: set(function_mapping_clean.get(code, []))
    for product_id, code in zip(
        products_check["PRODUCT_ID"],
        products_check["OVEN_FUNCTIONS"]
    )
}

# --------------------------------------------------
# 2. Independent requirement evaluation
# --------------------------------------------------

def independent_requirement_mask(requirement):

    characteristic = str(requirement["CHARACTERISTIC"]).strip()
    operator = str(requirement["OPERATOR"]).strip().upper()
    value = str(requirement["VALUE"]).strip()

    if characteristic not in products_check.columns:
        raise ValueError(
            f"Unknown characteristic: {characteristic}"
        )

    if operator == "EQUAL":

        return (
            products_check[characteristic] == value
        ).fillna(False)

    elif operator == "CONTAINS":

        if characteristic != "OVEN_FUNCTIONS":
            raise ValueError(
                f"Unsupported CONTAINS characteristic: {characteristic}"
            )

        return products_check["PRODUCT_ID"].map(
            lambda pid: value in product_function_sets[pid]
        )

    else:
        raise ValueError(
            f"Unsupported operator: {operator}"
        )


# --------------------------------------------------
# 3. Validate all customer requests
# --------------------------------------------------

validation_results = []
mismatch_details = []

for request_id, request_reqs in ground_truth.groupby(
    "REQUEST_ID", sort=False
):

    must_reqs = request_reqs[
        request_reqs["REQUIREMENT_TYPE"] == "MUST"
    ]

    preferred_reqs = request_reqs[
        request_reqs["REQUIREMENT_TYPE"] == "PREFERRED"
    ]

    # Independent MUST filtering
    eligible_mask = pd.Series(
        True,
        index=products_check.index
    )

    for _, req in must_reqs.iterrows():
        eligible_mask &= independent_requirement_mask(req)

    independent_eligible = products_check.loc[
        eligible_mask, "PRODUCT_ID"
    ]

    independent_ids = set(independent_eligible)

    # Original matching results
    original_ranking = rank_products_for_request(request_id)

    original_ids = set(
        original_ranking["PRODUCT_ID"].astype(str)
    )

    # Compare exact product sets
    false_inclusions = original_ids - independent_ids
    false_exclusions = independent_ids - original_ids

    eligibility_match = (
        original_ids == independent_ids
    )

    # Independent PREFERRED scoring
    score_match = True
    score_mismatches = []

    if independent_ids:

        independent_scores = pd.Series(
            0,
            index=products_check.index,
            dtype=int
        )

        for _, req in preferred_reqs.iterrows():
            independent_scores += (
                independent_requirement_mask(req).astype(int)
            )

        expected_scores = dict(zip(
            products_check.loc[
                eligible_mask, "PRODUCT_ID"
            ],
            independent_scores.loc[eligible_mask]
        ))

        actual_scores = dict(zip(
            original_ranking["PRODUCT_ID"],
            original_ranking["PREFERRED_SCORE"]
        ))

        score_mismatches = [
            pid
            for pid in independent_ids
            if expected_scores[pid] != actual_scores.get(pid)
        ]

        score_match = len(score_mismatches) == 0

    validation_results.append({
        "REQUEST_ID": request_id,
        "ORIGINAL_ELIGIBLE": len(original_ids),
        "INDEPENDENT_ELIGIBLE": len(independent_ids),
        "ELIGIBILITY_MATCH": eligibility_match,
        "SCORE_MATCH": score_match,
        "FALSE_INCLUSIONS": len(false_inclusions),
        "FALSE_EXCLUSIONS": len(false_exclusions),
        "SCORE_MISMATCHES": len(score_mismatches)
    })

    if not eligibility_match or not score_match:

        mismatch_details.append({
            "REQUEST_ID": request_id,
            "FALSE_INCLUSIONS": sorted(false_inclusions),
            "FALSE_EXCLUSIONS": sorted(false_exclusions),
            "SCORE_MISMATCHES": score_mismatches
        })


# --------------------------------------------------
# 4. Validation summary
# --------------------------------------------------

validation_df = pd.DataFrame(validation_results)

print("=== INDEPENDENT MATCHING VALIDATION ===")

print("Requests checked:", len(validation_df))

print(
    "Eligibility matches:",
    int(validation_df["ELIGIBILITY_MATCH"].sum())
)

print(
    "Preference score matches:",
    int(validation_df["SCORE_MATCH"].sum())
)

print(
    "Requests with discrepancies:",
    len(mismatch_details)
)

print(
    "Total false inclusions:",
    int(validation_df["FALSE_INCLUSIONS"].sum())
)

print(
    "Total false exclusions:",
    int(validation_df["FALSE_EXCLUSIONS"].sum())
)

print(
    "Total score mismatches:",
    int(validation_df["SCORE_MISMATCHES"].sum())
)

print("\n=== SAMPLE VALIDATION RESULTS ===")
display(validation_df.head(10))

if mismatch_details:
    print("\n=== DISCREPANCY DETAILS ===")
    display(pd.DataFrame(mismatch_details).head(10))
else:
    print("\nNo discrepancies detected.")

print("\nSTEP 176: COMPLETED")

=== INDEPENDENT MATCHING VALIDATION ===
Requests checked: 1967
Eligibility matches: 1967
Preference score matches: 1967
Requests with discrepancies: 0
Total false inclusions: 0
Total false exclusions: 0
Total score mismatches: 0

=== SAMPLE VALIDATION RESULTS ===


,REQUEST_ID,ORIGINAL_ELIGIBLE,INDEPENDENT_ELIGIBLE,ELIGIBILITY_MATCH,SCORE_MATCH,FALSE_INCLUSIONS,FALSE_EXCLUSIONS,SCORE_MISMATCHES
0,R0001,3,3,True,True,0,0,0
1,R0002,4,4,True,True,0,0,0
2,R0003,0,0,True,True,0,0,0
3,R0004,38,38,True,True,0,0,0
4,R0005,2,2,True,True,0,0,0
5,R0007,8,8,True,True,0,0,0
6,R0008,0,0,True,True,0,0,0
7,R0009,3,3,True,True,0,0,0
8,R0012,50,50,True,True,0,0,0
9,R0014,0,0,True,True,0,0,0



No discrepancies detected.

STEP 176: COMPLETED


## STEP 181 — Load Saved NLP Models

Load the trained NLP models saved in Notebook 04.

Objectives:
- Load the selected Linear SVM requirement extraction model.
- Restore its TF-IDF vectorizer, label mapping, and threshold.
- Load the Logistic Regression priority classification pipeline.
- Verify that all required model components are available.

No model training is performed in this step.

In [15]:
# STEP 181 — Load Saved NLP Models

from pathlib import Path
import joblib

MODEL_DIR = Path("../models")

nlp1_path = MODEL_DIR / "nlp1_requirement_extraction.joblib"
nlp2_path = MODEL_DIR / "nlp2_priority_classification.joblib"

assert nlp1_path.exists(), f"Missing model: {nlp1_path}"
assert nlp2_path.exists(), f"Missing model: {nlp2_path}"

# Load saved artifacts
nlp1_artifact = joblib.load(nlp1_path)
nlp2_artifact = joblib.load(nlp2_path)

# NLP 1 components
nlp1_vectorizer = nlp1_artifact["vectorizer"]
nlp1_model = nlp1_artifact["model"]
nlp1_labels = nlp1_artifact["labels"]
nlp1_threshold = nlp1_artifact["threshold"]

# NLP 2 components
nlp2_pipeline = nlp2_artifact["pipeline"]

print("=== NLP MODEL LOADING ===")

print("\nNLP 1 — Requirement Extraction")
print("Model:", type(nlp1_model).__name__)
print("Vectorizer:", type(nlp1_vectorizer).__name__)
print("Labels:", len(nlp1_labels))
print("Threshold:", nlp1_threshold)

print("\nNLP 2 — Priority Classification")
print("Pipeline:", type(nlp2_pipeline).__name__)
print("Classes:", nlp2_pipeline.classes_)

print("\nSTEP 181: COMPLETED")

=== NLP MODEL LOADING ===

NLP 1 — Requirement Extraction
Model: OneVsRestClassifier
Vectorizer: TfidfVectorizer
Labels: 99
Threshold: -0.3

NLP 2 — Priority Classification
Pipeline: Pipeline
Classes: [0 1]

STEP 181: COMPLETED


## STEP 182 — Test NLP 1 Requirement Extraction

This step tests the saved Linear SVM requirement extraction model
using an existing customer request.

Objectives:
- Load a customer request from the validated NLP dataset.
- Generate characteristic-value predictions using the saved model.
- Apply the selected decision threshold (-0.30).
- Compare predicted requirements with Ground Truth.

This is a functional integration test, not an independent
evaluation of model performance.

In [16]:
# STEP 182 — Inspect Customer Request Dataset

import pandas as pd

request_path = PROCESSED_DIR / "nlp_customer_requests.csv"

assert request_path.exists(), (
    f"Customer request file not found: {request_path}"
)

nlp_requests = pd.read_csv(request_path)

print("=== NLP CUSTOMER REQUEST DATASET ===")
print("Requests:", len(nlp_requests))

print("\n=== COLUMNS ===")
print(nlp_requests.columns.tolist())

print("\n=== SAMPLE REQUESTS ===")
display(nlp_requests.head(3))

print("\nSTEP 182: DATA INSPECTION COMPLETED")

=== NLP CUSTOMER REQUEST DATASET ===
Requests: 1967

=== COLUMNS ===
['REQUEST_ID', 'LLM_TEXT', 'INPUT_TOKENS', 'OUTPUT_TOKENS']

=== SAMPLE REQUESTS ===


,REQUEST_ID,LLM_TEXT,INPUT_TOKENS,OUTPUT_TOKENS
0,R0001,I'm looking for a steam oven that's exactly 45...,247,31
1,R0002,I'm looking for an oven with a white control p...,286,44
2,R0003,I'm looking for a built-in combination oven ab...,397,63



STEP 182: DATA INSPECTION COMPLETED


### STEP 182B — Test NLP 1 Predictions Against Ground Truth

Apply the saved Linear SVM model to a validated customer request.

Compare predicted characteristic-value labels with Ground Truth
to verify the NLP 1 inference pipeline.

This is a functional integration test, not a new model evaluation.

In [17]:
# STEP 182B — Test NLP 1 Predictions Against Ground Truth

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Select customer request
# --------------------------------------------------

sample_request_id = "R0001"

sample_request = nlp_requests.loc[
    nlp_requests["REQUEST_ID"] == sample_request_id
]

assert len(sample_request) == 1, (
    f"Expected one request for {sample_request_id}"
)

customer_text = sample_request["LLM_TEXT"].iloc[0]

print("=== CUSTOMER REQUEST ===")
print("Request ID:", sample_request_id)
print("Text:", customer_text)

# --------------------------------------------------
# 2. TF-IDF transformation
# --------------------------------------------------

X_sample = nlp1_vectorizer.transform([customer_text])

# --------------------------------------------------
# 3. Linear SVM prediction
# --------------------------------------------------

decision_scores = np.asarray(
    nlp1_model.decision_function(X_sample)
).reshape(-1)

assert len(decision_scores) == len(nlp1_labels), (
    "Model output and label mapping dimensions differ."
)

predicted_mask = decision_scores >= nlp1_threshold

predicted_labels = [
    label
    for label, selected in zip(nlp1_labels, predicted_mask)
    if selected
]

# --------------------------------------------------
# 4. Ground Truth comparison
# --------------------------------------------------

gt_sample = ground_truth.loc[
    ground_truth["REQUEST_ID"] == sample_request_id
].copy()

gt_sample["LABEL"] = (
    gt_sample["CHARACTERISTIC"].astype(str).str.strip()
    + "="
    + gt_sample["VALUE"].astype(str).str.strip()
)

true_labels = gt_sample["LABEL"].tolist()

# --------------------------------------------------
# 5. Results
# --------------------------------------------------

print("\n=== NLP 1 PREDICTED LABELS ===")
print(predicted_labels)

print("\n=== GROUND TRUTH LABELS ===")
print(true_labels)

predicted_set = set(predicted_labels)
true_set = set(true_labels)

print("\n=== COMPARISON ===")
print("Correct predictions:", sorted(predicted_set & true_set))
print("Missing requirements:", sorted(true_set - predicted_set))
print("Extra predictions:", sorted(predicted_set - true_set))
print("Exact match:", predicted_set == true_set)

print("\nSTEP 182B: COMPLETED")

=== CUSTOMER REQUEST ===
Request ID: R0001
Text: I'm looking for a steam oven that's exactly 45 cm wide.



=== NLP 1 PREDICTED LABELS ===
['OVEN_TYPE__V002', 'WIDTH_CM__V001']

=== GROUND TRUTH LABELS ===
['OVEN_TYPE=V002', 'WIDTH_CM=V001']

=== COMPARISON ===
Correct predictions: []
Missing requirements: ['OVEN_TYPE=V002', 'WIDTH_CM=V001']
Extra predictions: ['OVEN_TYPE__V002', 'WIDTH_CM__V001']
Exact match: False

STEP 182B: COMPLETED


### STEP 182C — Normalize NLP Label Format

The NLP model uses CHARACTERISTIC__VALUE labels, while the
Ground Truth comparison initially used CHARACTERISTIC=VALUE.

This step normalizes both representations and repeats the
comparison without changing the trained model.

In [18]:
# STEP 182C — Correct NLP Label Comparison

# Convert Ground Truth to the model's label format
true_labels_normalized = (
    gt_sample["CHARACTERISTIC"].astype(str).str.strip()
    + "__"
    + gt_sample["VALUE"].astype(str).str.strip()
).tolist()

predicted_set = set(predicted_labels)
true_set = set(true_labels_normalized)

print("=== NLP 1 LABEL COMPARISON ===")

print("Predicted:", sorted(predicted_set))
print("Ground Truth:", sorted(true_set))

print("\nCorrect predictions:", sorted(predicted_set & true_set))
print("Missing requirements:", sorted(true_set - predicted_set))
print("Extra predictions:", sorted(predicted_set - true_set))

exact_match = predicted_set == true_set

print("\nExact Match:", exact_match)

assert exact_match, "R0001 predictions do not match Ground Truth."

print("\nSTEP 182C: COMPLETED")

=== NLP 1 LABEL COMPARISON ===
Predicted: ['OVEN_TYPE__V002', 'WIDTH_CM__V001']
Ground Truth: ['OVEN_TYPE__V002', 'WIDTH_CM__V001']

Correct predictions: ['OVEN_TYPE__V002', 'WIDTH_CM__V001']
Missing requirements: []
Extra predictions: []

Exact Match: True

STEP 182C: COMPLETED


## STEP 183 — Convert NLP Predictions into Structured Requirements

Convert NLP 1 predictions into the structured format required
by the Product Matching algorithm.

Objectives:
- Split predicted labels into characteristic and value codes.
- Assign matching operators.
- Validate predicted characteristics and values.
- Prepare structured requirements for NLP 2 priority classification.

Requirement priorities will be predicted in the next step.

In [19]:
# STEP 183 — Convert NLP Predictions into Structured Requirements

import pandas as pd

structured_requirements = []

for label in predicted_labels:

    # Split the model label into characteristic and value
    characteristic, value = label.rsplit("__", 1)

    # Determine matching operator
    if characteristic == "OVEN_FUNCTIONS":
        operator = "CONTAINS"
    else:
        operator = "EQUAL"

    # Validate characteristic
    assert characteristic in product_master.columns, (
        f"Unknown characteristic: {characteristic}"
    )

    # Validate value against product data or function mapping
    if operator == "EQUAL":

        valid_values = set(
            product_master[characteristic]
            .dropna()
            .astype(str)
            .str.strip()
        )

    else:

        valid_values = {
            function
            for functions in function_mapping_clean.values()
            for function in functions
        }

    assert value in valid_values, (
        f"Invalid value for {characteristic}: {value}"
    )

    structured_requirements.append({
        "REQUEST_ID": sample_request_id,
        "CHARACTERISTIC": characteristic,
        "VALUE": value,
        "OPERATOR": operator,
        "REQUIREMENT_TYPE": None
    })

predicted_requirements = pd.DataFrame(
    structured_requirements,
    columns=[
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
)

print("=== STRUCTURED NLP REQUIREMENTS ===")

display(predicted_requirements)

print("\nSTEP 183: COMPLETED")

=== STRUCTURED NLP REQUIREMENTS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,R0001,OVEN_TYPE,V002,EQUAL,None
1,R0001,WIDTH_CM,V001,EQUAL,None



STEP 183: COMPLETED


## STEP 185 — NLP 2 Priority Prediction

Predict MUST/PREFERRED priorities for NLP 1 requirements.

- Convert characteristic-value codes into readable descriptions.
- Reconstruct the input format used during NLP 2 training.
- Predict requirement priorities using the saved Logistic Regression pipeline.
- Compare predictions with Ground Truth.

Ground Truth priority labels are used only for evaluation.

In [20]:
# STEP 185 — NLP 2 Priority Prediction

import pandas as pd

# --------------------------------------------------
# 1. Load requirement descriptions
# --------------------------------------------------

description_path = PROCESSED_DIR / "nlp_ground_truth.csv"

description_data = pd.read_csv(description_path)

required_columns = {
    "CHARACTERISTIC",
    "VALUE",
    "VALUE_DEFINITION"
}

assert required_columns.issubset(description_data.columns), (
    f"Missing columns: {required_columns - set(description_data.columns)}"
)

# Build characteristic-value description dictionary
description_lookup = (
    description_data[
        ["CHARACTERISTIC", "VALUE", "VALUE_DEFINITION"]
    ]
    .dropna()
    .drop_duplicates()
)

# Check for conflicting descriptions
description_counts = (
    description_lookup
    .groupby(["CHARACTERISTIC", "VALUE"])["VALUE_DEFINITION"]
    .nunique()
)

conflicts = description_counts[description_counts > 1]

assert conflicts.empty, (
    "Conflicting VALUE_DEFINITION descriptions detected."
)

description_map = {
    (row.CHARACTERISTIC, row.VALUE): row.VALUE_DEFINITION
    for row in description_lookup.itertuples(index=False)
}

# --------------------------------------------------
# 2. Prepare NLP 2 input
# --------------------------------------------------

priority_input = predicted_requirements.copy()

priority_input["VALUE_DEFINITION"] = priority_input.apply(
    lambda row: description_map.get(
        (row["CHARACTERISTIC"], row["VALUE"])
    ),
    axis=1
)

assert priority_input["VALUE_DEFINITION"].notna().all(), (
    "Missing requirement descriptions."
)

priority_input["REQUIREMENT_DESCRIPTION"] = (
    priority_input["CHARACTERISTIC"]
    + " = "
    + priority_input["VALUE_DEFINITION"].astype(str)
)

priority_input["NLP2_TEXT"] = (
    "CUSTOMER REQUEST: "
    + customer_text
    + " REQUIREMENT: "
    + priority_input["REQUIREMENT_DESCRIPTION"]
)

# --------------------------------------------------
# 3. Predict priority
# --------------------------------------------------

priority_predictions = nlp2_pipeline.predict(
    priority_input["NLP2_TEXT"].tolist()
)

priority_mapping = {
    0: "PREFERRED",
    1: "MUST"
}

priority_input["PREDICTED_PRIORITY"] = [
    priority_mapping[int(value)]
    for value in priority_predictions
]

# --------------------------------------------------
# 4. Compare with Ground Truth
# --------------------------------------------------

gt_priorities = ground_truth[
    ground_truth["REQUEST_ID"] == sample_request_id
][
    ["CHARACTERISTIC", "VALUE", "REQUIREMENT_TYPE"]
].copy()

gt_priorities = gt_priorities.rename(
    columns={"REQUIREMENT_TYPE": "ACTUAL_PRIORITY"}
)

priority_comparison = priority_input.merge(
    gt_priorities,
    on=["CHARACTERISTIC", "VALUE"],
    how="left",
    validate="one_to_one"
)

priority_comparison["PRIORITY_MATCH"] = (
    priority_comparison["PREDICTED_PRIORITY"]
    == priority_comparison["ACTUAL_PRIORITY"]
)

# --------------------------------------------------
# 5. Results
# --------------------------------------------------

print("=== NLP 2 PRIORITY PREDICTIONS ===")

display(
    priority_comparison[
        [
            "CHARACTERISTIC",
            "VALUE",
            "REQUIREMENT_DESCRIPTION",
            "PREDICTED_PRIORITY",
            "ACTUAL_PRIORITY",
            "PRIORITY_MATCH"
        ]
    ]
)

print("\n=== PRIORITY COMPARISON ===")

print(
    "Correct predictions:",
    int(priority_comparison["PRIORITY_MATCH"].sum())
)

print(
    "Total requirements:",
    len(priority_comparison)
)

print("\nSTEP 185: COMPLETED")

=== NLP 2 PRIORITY PREDICTIONS ===


,CHARACTERISTIC,VALUE,REQUIREMENT_DESCRIPTION,PREDICTED_PRIORITY,ACTUAL_PRIORITY,PRIORITY_MATCH
0,OVEN_TYPE,V002,OVEN_TYPE = STEAM,MUST,MUST,True
1,WIDTH_CM,V001,WIDTH_CM = 45,MUST,MUST,True



=== PRIORITY COMPARISON ===
Correct predictions: 2
Total requirements: 2

STEP 185: COMPLETED


## STEP 186 — NLP-Based Product Matching

Integrate NLP 1 requirement extraction and NLP 2 priority
classification with the Product Matching algorithm.

Objectives:
- Build structured requirements from NLP predictions.
- Filter products using predicted MUST requirements.
- Rank eligible products using predicted PREFERRED requirements.
- Compare NLP-based matching results with Ground Truth matching.

Ground Truth is used only for evaluation.

In [21]:
# STEP 186 — NLP-Based Product Matching

import pandas as pd

# --------------------------------------------------
# 1. Prepare NLP-predicted requirements
# --------------------------------------------------

nlp_requirements = priority_comparison[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "PREDICTED_PRIORITY"
    ]
].copy()

nlp_requirements = nlp_requirements.rename(
    columns={"PREDICTED_PRIORITY": "REQUIREMENT_TYPE"}
)

print("=== NLP-PREDICTED REQUIREMENTS ===")
display(nlp_requirements)

# --------------------------------------------------
# 2. Independent matching using NLP predictions
# --------------------------------------------------

eligible_mask = pd.Series(
    True,
    index=products_check.index
)

must_requirements = nlp_requirements[
    nlp_requirements["REQUIREMENT_TYPE"] == "MUST"
]

preferred_requirements = nlp_requirements[
    nlp_requirements["REQUIREMENT_TYPE"] == "PREFERRED"
]

# Apply MUST conditions
for _, requirement in must_requirements.iterrows():
    eligible_mask &= independent_requirement_mask(requirement)

eligible_products = products_check.loc[
    eligible_mask
].copy()

# --------------------------------------------------
# 3. Calculate PREFERRED scores
# --------------------------------------------------

eligible_products["PREFERRED_SCORE"] = 0

for _, requirement in preferred_requirements.iterrows():

    preference_mask = independent_requirement_mask(requirement)

    eligible_products["PREFERRED_SCORE"] += (
        preference_mask.loc[eligible_products.index].astype(int)
    )

eligible_products = eligible_products.sort_values(
    ["PREFERRED_SCORE", "PRODUCT_ID"],
    ascending=[False, True]
)

# --------------------------------------------------
# 4. Compare with Ground Truth matching
# --------------------------------------------------

ground_truth_ranking = rank_products_for_request(
    sample_request_id
)

nlp_product_ids = set(
    eligible_products["PRODUCT_ID"]
)

ground_truth_product_ids = set(
    ground_truth_ranking["PRODUCT_ID"]
)

print("\n=== PRODUCT MATCHING COMPARISON ===")

print("Request ID:", sample_request_id)
print("NLP eligible products:", len(nlp_product_ids))
print("Ground Truth eligible products:", len(ground_truth_product_ids))

print(
    "Product sets match:",
    nlp_product_ids == ground_truth_product_ids
)

print("\n=== NLP-BASED RECOMMENDED PRODUCTS ===")

display(
    eligible_products[
        ["PRODUCT_ID", "PREFERRED_SCORE"]
    ].head(10)
)

print("\nSTEP 186: COMPLETED")

=== NLP-PREDICTED REQUIREMENTS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,R0001,OVEN_TYPE,V002,EQUAL,MUST
1,R0001,WIDTH_CM,V001,EQUAL,MUST



=== PRODUCT MATCHING COMPARISON ===
Request ID: R0001
NLP eligible products: 3
Ground Truth eligible products: 3
Product sets match: True

=== NLP-BASED RECOMMENDED PRODUCTS ===


,PRODUCT_ID,PREFERRED_SCORE
74,P0075,0
139,P0140,0
448,P0449,0



STEP 186: COMPLETED


In [22]:
# STEP 188A — Inspect Original NLP Test Split

import json
from pathlib import Path

notebook_path = Path("04_nlp_model_training.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook_04 = json.load(f)

for cell in notebook_04["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if "Step 144 -" not in source[:150]:
        continue

    lines = source.splitlines()

    print("=== ORIGINAL NLP TEST SPLIT ===")

    for i in range(74, min(125, len(lines))):
        print(f"{i + 1}: {lines[i]}")

    break

print("\nSTEP 188A: COMPLETED")

=== ORIGINAL NLP TEST SPLIT ===
75: 
76: # First split: 70% training, 30% temporary
77: 
78: splitter_1 = MultilabelStratifiedShuffleSplit(
79:     n_splits=1,
80:     test_size=0.30,
81:     random_state=42
82: )
83: 
84: train_idx, temp_idx = next(
85:     splitter_1.split(X, y)
86: )
87: 
88: 
89: # Second split: temporary data into validation and test
90: 
91: splitter_2 = MultilabelStratifiedShuffleSplit(
92:     n_splits=1,
93:     test_size=0.50,
94:     random_state=42
95: )
96: 
97: val_relative_idx, test_relative_idx = next(
98:     splitter_2.split(
99:         X.iloc[temp_idx],
100:         y.iloc[temp_idx]
101:     )
102: )
103: 
104: val_idx = temp_idx[val_relative_idx]
105: test_idx = temp_idx[test_relative_idx]
106: 
107: 
108: # Create datasets
109: 
110: X_train = X.iloc[train_idx]
111: X_val = X.iloc[val_idx]
112: X_test = X.iloc[test_idx]
113: 
114: y_train = y.iloc[train_idx]
115: y_val = y.iloc[val_idx]
116: y_test = y.iloc[test_idx]
117: 
118: 
119: # Validate re

## STEP 188C — Prepare End-to-End Test Dataset

Load the original held-out test request IDs exported from Notebook 04.

Objectives:
- Select the original 298 test customer requests.
- Verify that all test requests have corresponding customer text.
- Prepare Ground Truth records for independent evaluation.
- Preserve the original train/validation/test split.

Ground Truth is reserved for evaluation and is not used as
input to the NLP prediction models.

In [23]:
# STEP 188C — Prepare End-to-End Test Dataset

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Load original test request IDs
# --------------------------------------------------

test_ids_path = PROCESSED_DIR / "nlp_test_request_ids.csv"

assert test_ids_path.exists(), (
    f"Test ID file not found: {test_ids_path}"
)

test_request_ids = pd.read_csv(test_ids_path)

assert test_request_ids["REQUEST_ID"].is_unique
assert test_request_ids["REQUEST_ID"].notna().all()

# --------------------------------------------------
# 2. Select test customer requests
# --------------------------------------------------

test_requests = nlp_requests.merge(
    test_request_ids,
    on="REQUEST_ID",
    how="inner",
    validate="one_to_one"
)

assert len(test_requests) == len(test_request_ids), (
    "Some test requests are missing from the customer dataset."
)

assert test_requests["LLM_TEXT"].notna().all()

test_requests = (
    test_requests[
        ["REQUEST_ID", "LLM_TEXT"]
    ]
    .sort_values("REQUEST_ID")
    .reset_index(drop=True)
)

# --------------------------------------------------
# 3. Prepare Ground Truth for evaluation
# --------------------------------------------------

test_ground_truth = ground_truth[
    ground_truth["REQUEST_ID"].isin(
        test_request_ids["REQUEST_ID"]
    )
].copy()

assert set(test_requests["REQUEST_ID"]) == set(
    test_ground_truth["REQUEST_ID"]
), "Some test requests have no Ground Truth records."

# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

print("=== END-TO-END TEST DATASET ===")

print("Test requests:", len(test_requests))
print("Unique request IDs:", test_requests["REQUEST_ID"].nunique())
print("Ground Truth requirements:", len(test_ground_truth))

print("\n=== REQUIREMENT OPERATORS ===")
print(test_ground_truth["OPERATOR"].value_counts())

print("\n=== REQUIREMENT PRIORITIES ===")
print(test_ground_truth["REQUIREMENT_TYPE"].value_counts())

print("\n=== SAMPLE TEST REQUESTS ===")
display(test_requests.head(5))

print("\nSTEP 188C: COMPLETED")

=== END-TO-END TEST DATASET ===
Test requests: 298
Unique request IDs: 298
Ground Truth requirements: 1228

=== REQUIREMENT OPERATORS ===
OPERATOR
EQUAL       797
CONTAINS    431
Name: count, dtype: int64

=== REQUIREMENT PRIORITIES ===
REQUIREMENT_TYPE
MUST         896
PREFERRED    332
Name: count, dtype: int64

=== SAMPLE TEST REQUESTS ===


,REQUEST_ID,LLM_TEXT
0,R0002,I'm looking for an oven with a white control p...
1,R0003,I'm looking for a built-in combination oven ab...
2,R0009,I'm looking for a 45 cm-wide oven with a white...
3,R0015,I'm looking for a stainless steel oven (no rot...
4,R0018,I need a built-in oven that definitely comes w...



STEP 188C: COMPLETED


## STEP 188D — Batch NLP 1 Requirement Extraction

Evaluate the saved NLP 1 model on the original held-out test set.

Objectives:
- Predict characteristic-value labels for 298 customer requests.
- Apply the selected Linear SVM decision threshold.
- Convert predictions into structured requirements.
- Calculate requirement extraction performance.

Ground Truth is used only for evaluation.

In [24]:
# STEP 188D — Batch NLP 1 Requirement Extraction

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# --------------------------------------------------
# 1. Transform customer requests
# --------------------------------------------------

X_test_nlp = nlp1_vectorizer.transform(
    test_requests["LLM_TEXT"].astype(str)
)

# --------------------------------------------------
# 2. Generate Linear SVM decision scores
# --------------------------------------------------

decision_scores = np.asarray(
    nlp1_model.decision_function(X_test_nlp)
)

assert decision_scores.shape == (
    len(test_requests),
    len(nlp1_labels)
), "Unexpected NLP 1 prediction dimensions."

# Apply selected threshold
prediction_matrix = (
    decision_scores >= nlp1_threshold
).astype(int)

# --------------------------------------------------
# 3. Convert predictions into structured requirements
# --------------------------------------------------

predicted_records = []

for row_index, request_id in enumerate(
    test_requests["REQUEST_ID"]
):

    selected_indices = np.flatnonzero(
        prediction_matrix[row_index]
    )

    for label_index in selected_indices:

        label = nlp1_labels[label_index]

        characteristic, value = label.rsplit("__", 1)

        operator = (
            "CONTAINS"
            if characteristic == "OVEN_FUNCTIONS"
            else "EQUAL"
        )

        predicted_records.append({
            "REQUEST_ID": request_id,
            "CHARACTERISTIC": characteristic,
            "VALUE": value,
            "OPERATOR": operator
        })

batch_nlp1_requirements = pd.DataFrame(
    predicted_records,
    columns=[
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR"
    ]
)

# --------------------------------------------------
# 4. Prepare Ground Truth label matrix
# --------------------------------------------------

ground_truth_labels = (
    test_ground_truth["CHARACTERISTIC"].astype(str)
    + "__"
    + test_ground_truth["VALUE"].astype(str)
)

ground_truth_matrix = pd.DataFrame(
    0,
    index=test_requests["REQUEST_ID"],
    columns=nlp1_labels,
    dtype=int
)

for request_id, label in zip(
    test_ground_truth["REQUEST_ID"],
    ground_truth_labels
):

    assert label in ground_truth_matrix.columns, (
        f"Unknown Ground Truth label: {label}"
    )

    ground_truth_matrix.loc[
        request_id, label
    ] = 1

y_true = ground_truth_matrix.to_numpy()
y_pred = prediction_matrix

# --------------------------------------------------
# 5. Evaluate NLP 1
# --------------------------------------------------

micro_precision = precision_score(
    y_true, y_pred,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    y_true, y_pred,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    y_true, y_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

exact_match = np.mean(
    np.all(y_true == y_pred, axis=1)
)

# --------------------------------------------------
# 6. Results
# --------------------------------------------------

print("=== BATCH NLP 1 EVALUATION ===")

print("Test requests:", len(test_requests))
print("Predicted requirements:", len(batch_nlp1_requirements))
print("Ground Truth requirements:", len(test_ground_truth))

print("\n=== NLP 1 PERFORMANCE ===")

print(f"Micro Precision: {micro_precision:.4f}")
print(f"Micro Recall:    {micro_recall:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Exact Match:     {exact_match:.4f}")

print("\n=== SAMPLE PREDICTED REQUIREMENTS ===")

display(batch_nlp1_requirements.head(10))

print("\nSTEP 188D: COMPLETED")

=== BATCH NLP 1 EVALUATION ===
Test requests: 298
Predicted requirements: 1198
Ground Truth requirements: 1228

=== NLP 1 PERFORMANCE ===
Micro Precision: 0.9416
Micro Recall:    0.9186
Micro F1:        0.9299
Macro F1:        0.9130
Exact Match:     0.6477

=== SAMPLE PREDICTED REQUIREMENTS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR
0,R0002,KNOB_COLOR,V002,EQUAL
1,R0002,PACKAGING_TYPE,V001,EQUAL
2,R0003,BAKING_TRAY,V003,EQUAL
3,R0003,KNOB_TYPE,V002,EQUAL
4,R0003,OVEN_TYPE,V003,EQUAL
5,R0003,PANEL_TYPE,V001,EQUAL
6,R0003,WIDTH_CM,V002,EQUAL
7,R0009,HANDLE_COLOR,V002,EQUAL
8,R0009,TURNSPIT,V001,EQUAL
9,R0009,WIDTH_CM,V001,EQUAL



STEP 188D: COMPLETED


## STEP 188E — Batch NLP 2 Priority Prediction

Apply the saved priority classification model to all requirements
predicted by NLP 1 in the held-out test set.

Objectives:
- Construct the NLP 2 input using customer text and readable
  requirement descriptions.
- Predict MUST/PREFERRED priorities.
- Compare predicted priorities with Ground Truth where the
  characteristic-value requirement is correctly identified.
- Prepare the predicted requirements for Product Matching.

Ground Truth priority labels are used only for evaluation.

In [25]:
# STEP 188E — Batch NLP 2 Priority Prediction

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# --------------------------------------------------
# 1. Prepare NLP 1 predicted requirements
# --------------------------------------------------

batch_nlp2_input = batch_nlp1_requirements.copy()

# Attach customer text
batch_nlp2_input = batch_nlp2_input.merge(
    test_requests,
    on="REQUEST_ID",
    how="left",
    validate="many_to_one"
)

assert batch_nlp2_input["LLM_TEXT"].notna().all(), (
    "Missing customer text."
)

# --------------------------------------------------
# 2. Attach readable requirement descriptions
# --------------------------------------------------

batch_nlp2_input["VALUE_DEFINITION"] = (
    batch_nlp2_input.apply(
        lambda row: description_map.get(
            (row["CHARACTERISTIC"], row["VALUE"])
        ),
        axis=1
    )
)

missing_descriptions = batch_nlp2_input[
    batch_nlp2_input["VALUE_DEFINITION"].isna()
]

print("=== DESCRIPTION VALIDATION ===")
print("Predicted requirements:", len(batch_nlp2_input))
print("Missing descriptions:", len(missing_descriptions))

assert missing_descriptions.empty, (
    "Some predicted requirements have no VALUE_DEFINITION."
)

# --------------------------------------------------
# 3. Reconstruct NLP 2 training input format
# --------------------------------------------------

batch_nlp2_input["REQUIREMENT_DESCRIPTION"] = (
    batch_nlp2_input["CHARACTERISTIC"]
    + " = "
    + batch_nlp2_input["VALUE_DEFINITION"].astype(str)
)

batch_nlp2_input["NLP2_TEXT"] = (
    "CUSTOMER REQUEST: "
    + batch_nlp2_input["LLM_TEXT"].astype(str)
    + " REQUIREMENT: "
    + batch_nlp2_input["REQUIREMENT_DESCRIPTION"]
)

# --------------------------------------------------
# 4. Predict MUST / PREFERRED
# --------------------------------------------------

priority_predictions = nlp2_pipeline.predict(
    batch_nlp2_input["NLP2_TEXT"].tolist()
)

priority_mapping = nlp2_artifact["class_mapping"]

batch_nlp2_input["REQUIREMENT_TYPE"] = [
    priority_mapping[int(value)]
    for value in priority_predictions
]

# --------------------------------------------------
# 5. Compare with Ground Truth
# --------------------------------------------------

gt_priority_reference = test_ground_truth[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "REQUIREMENT_TYPE"
    ]
].copy()

gt_priority_reference = gt_priority_reference.rename(
    columns={
        "REQUIREMENT_TYPE": "ACTUAL_PRIORITY"
    }
)

priority_evaluation = batch_nlp2_input.merge(
    gt_priority_reference,
    on=["REQUEST_ID", "CHARACTERISTIC", "VALUE"],
    how="left",
    validate="one_to_one"
)

# Evaluate only correctly extracted requirements
matched_requirements = priority_evaluation[
    priority_evaluation["ACTUAL_PRIORITY"].notna()
].copy()

priority_accuracy = accuracy_score(
    matched_requirements["ACTUAL_PRIORITY"],
    matched_requirements["REQUIREMENT_TYPE"]
)

priority_macro_f1 = f1_score(
    matched_requirements["ACTUAL_PRIORITY"],
    matched_requirements["REQUIREMENT_TYPE"],
    average="macro"
)

# --------------------------------------------------
# 6. Prepare Product Matching input
# --------------------------------------------------

batch_predicted_requirements = batch_nlp2_input[
    [
        "REQUEST_ID",
        "CHARACTERISTIC",
        "VALUE",
        "OPERATOR",
        "REQUIREMENT_TYPE"
    ]
].copy()

# --------------------------------------------------
# 7. Results
# --------------------------------------------------

print("\n=== BATCH NLP 2 RESULTS ===")

print("Test requests:", len(test_requests))
print("Predicted requirements:", len(batch_predicted_requirements))

print("\n=== PRIORITY DISTRIBUTION ===")
print(
    batch_predicted_requirements[
        "REQUIREMENT_TYPE"
    ].value_counts()
)

print("\n=== PRIORITY EVALUATION ===")

print("Matched requirements:", len(matched_requirements))
print(
    "Unmatched predicted requirements:",
    len(priority_evaluation) - len(matched_requirements)
)

print(f"Priority Accuracy: {priority_accuracy:.4f}")
print(f"Priority Macro F1: {priority_macro_f1:.4f}")

print("\n=== SAMPLE PREDICTED REQUIREMENTS ===")
display(batch_predicted_requirements.head(10))

print("\nSTEP 188E: COMPLETED")

=== DESCRIPTION VALIDATION ===
Predicted requirements: 1198
Missing descriptions: 0

=== BATCH NLP 2 RESULTS ===
Test requests: 298
Predicted requirements: 1198

=== PRIORITY DISTRIBUTION ===
REQUIREMENT_TYPE
MUST         632
PREFERRED    566
Name: count, dtype: int64

=== PRIORITY EVALUATION ===
Matched requirements: 1128
Unmatched predicted requirements: 70
Priority Accuracy: 0.6348
Priority Macro F1: 0.6073

=== SAMPLE PREDICTED REQUIREMENTS ===


,REQUEST_ID,CHARACTERISTIC,VALUE,OPERATOR,REQUIREMENT_TYPE
0,R0002,KNOB_COLOR,V002,EQUAL,MUST
1,R0002,PACKAGING_TYPE,V001,EQUAL,MUST
2,R0003,BAKING_TRAY,V003,EQUAL,PREFERRED
3,R0003,KNOB_TYPE,V002,EQUAL,PREFERRED
4,R0003,OVEN_TYPE,V003,EQUAL,MUST
5,R0003,PANEL_TYPE,V001,EQUAL,MUST
6,R0003,WIDTH_CM,V002,EQUAL,MUST
7,R0009,HANDLE_COLOR,V002,EQUAL,MUST
8,R0009,TURNSPIT,V001,EQUAL,PREFERRED
9,R0009,WIDTH_CM,V001,EQUAL,MUST



STEP 188E: COMPLETED


## STEP 188F — End-to-End Product Matching Evaluation

Evaluate product matching performance using NLP-predicted requirements.

For each held-out customer request:

1. Filter products using NLP-predicted MUST requirements.
2. Independently filter products using Ground Truth MUST requirements.
3. Compare the resulting eligible product sets.
4. Measure product eligibility precision, recall, and exact match.
5. Evaluate agreement on product availability.

PREFERRED requirements are not used for product eligibility.
They will be evaluated separately during product ranking.

No Ground Truth information is used to correct NLP predictions.

In [26]:
# STEP 188F — End-to-End Product Matching Evaluation

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Validate inputs
# --------------------------------------------------

assert "independent_requirement_mask" in globals()
assert "products_check" in globals()

assert products_check["PRODUCT_ID"].is_unique

assert set(batch_predicted_requirements["REQUEST_ID"]).issubset(
    set(test_requests["REQUEST_ID"])
)

assert set(test_ground_truth["REQUEST_ID"]).issubset(
    set(test_requests["REQUEST_ID"])
)

# --------------------------------------------------
# 2. Function: identify eligible products
# --------------------------------------------------

def get_eligible_product_ids(requirements):

    eligible_mask = pd.Series(
        True,
        index=products_check.index
    )

    must_requirements = requirements[
        requirements["REQUIREMENT_TYPE"] == "MUST"
    ]

    for _, requirement in must_requirements.iterrows():

        requirement_mask = independent_requirement_mask(
            requirement
        )

        eligible_mask &= requirement_mask

    return set(
        products_check.loc[
            eligible_mask, "PRODUCT_ID"
        ]
    )


# --------------------------------------------------
# 3. Evaluate each test request
# --------------------------------------------------

evaluation_records = []

total_true_positive = 0
total_false_positive = 0
total_false_negative = 0

for request_id in test_requests["REQUEST_ID"]:

    # NLP-predicted requirements
    predicted_req = batch_predicted_requirements[
        batch_predicted_requirements["REQUEST_ID"] == request_id
    ]

    # Ground Truth requirements
    actual_req = test_ground_truth[
        test_ground_truth["REQUEST_ID"] == request_id
    ]

    # Product matching
    predicted_products = get_eligible_product_ids(
        predicted_req
    )

    actual_products = get_eligible_product_ids(
        actual_req
    )

    # Compare product sets
    true_positive = len(
        predicted_products & actual_products
    )

    false_positive = len(
        predicted_products - actual_products
    )

    false_negative = len(
        actual_products - predicted_products
    )

    total_true_positive += true_positive
    total_false_positive += false_positive
    total_false_negative += false_negative

    product_set_match = (
        predicted_products == actual_products
    )

    predicted_has_match = len(predicted_products) > 0
    actual_has_match = len(actual_products) > 0

    evaluation_records.append({

        "REQUEST_ID": request_id,

        "PREDICTED_REQUIREMENTS": len(predicted_req),
        "ACTUAL_REQUIREMENTS": len(actual_req),

        "PREDICTED_ELIGIBLE": len(predicted_products),
        "ACTUAL_ELIGIBLE": len(actual_products),

        "TRUE_POSITIVE": true_positive,
        "FALSE_POSITIVE": false_positive,
        "FALSE_NEGATIVE": false_negative,

        "PRODUCT_SET_MATCH": product_set_match,

        "PREDICTED_HAS_MATCH": predicted_has_match,
        "ACTUAL_HAS_MATCH": actual_has_match,

        "AVAILABILITY_AGREEMENT": (
            predicted_has_match == actual_has_match
        )

    })


# --------------------------------------------------
# 4. Create evaluation dataframe
# --------------------------------------------------

product_matching_evaluation = pd.DataFrame(
    evaluation_records
)

# --------------------------------------------------
# 5. Calculate overall metrics
# --------------------------------------------------

product_set_exact_match = (
    product_matching_evaluation[
        "PRODUCT_SET_MATCH"
    ].mean()
)

availability_agreement = (
    product_matching_evaluation[
        "AVAILABILITY_AGREEMENT"
    ].mean()
)

precision_denominator = (
    total_true_positive + total_false_positive
)

recall_denominator = (
    total_true_positive + total_false_negative
)

eligibility_precision = (
    total_true_positive / precision_denominator
    if precision_denominator > 0
    else np.nan
)

eligibility_recall = (
    total_true_positive / recall_denominator
    if recall_denominator > 0
    else np.nan
)

eligibility_f1 = (
    2 * eligibility_precision * eligibility_recall
    / (eligibility_precision + eligibility_recall)
    if (
        precision_denominator > 0
        and recall_denominator > 0
        and eligibility_precision + eligibility_recall > 0
    )
    else np.nan
)

# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("=== END-TO-END PRODUCT MATCHING EVALUATION ===")

print("Test requests:", len(product_matching_evaluation))
print("Product portfolio:", len(products_check))

print("\n=== PRODUCT ELIGIBILITY PERFORMANCE ===")

print(
    f"Product Set Exact Match: "
    f"{product_set_exact_match:.4f}"
)

print(
    f"Eligibility Precision: "
    f"{eligibility_precision:.4f}"
)

print(
    f"Eligibility Recall: "
    f"{eligibility_recall:.4f}"
)

print(
    f"Eligibility F1: "
    f"{eligibility_f1:.4f}"
)

print(
    f"Availability Agreement: "
    f"{availability_agreement:.4f}"
)

print("\n=== PRODUCT MATCHING ERRORS ===")

print("True positive product pairs:", total_true_positive)
print("False positive product pairs:", total_false_positive)
print("False negative product pairs:", total_false_negative)

print("\n=== REQUEST-LEVEL SUMMARY ===")

print(
    "Exact product set matches:",
    int(
        product_matching_evaluation[
            "PRODUCT_SET_MATCH"
        ].sum()
    )
)

print(
    "Availability agreements:",
    int(
        product_matching_evaluation[
            "AVAILABILITY_AGREEMENT"
        ].sum()
    )
)

print("\n=== SAMPLE RESULTS ===")

display(
    product_matching_evaluation.head(10)
)

print("\nSTEP 188F: COMPLETED")

=== END-TO-END PRODUCT MATCHING EVALUATION ===
Test requests: 298
Product portfolio: 500

=== PRODUCT ELIGIBILITY PERFORMANCE ===
Product Set Exact Match: 0.3154
Eligibility Precision: 0.2758
Eligibility Recall: 0.8657
Eligibility F1: 0.4183
Availability Agreement: 0.9564

=== PRODUCT MATCHING ERRORS ===
True positive product pairs: 17011
False positive product pairs: 44668
False negative product pairs: 2639

=== REQUEST-LEVEL SUMMARY ===
Exact product set matches: 94
Availability agreements: 285

=== SAMPLE RESULTS ===


,REQUEST_ID,PREDICTED_REQUIREMENTS,ACTUAL_REQUIREMENTS,PREDICTED_ELIGIBLE,ACTUAL_ELIGIBLE,TRUE_POSITIVE,FALSE_POSITIVE,FALSE_NEGATIVE,PRODUCT_SET_MATCH,PREDICTED_HAS_MATCH,ACTUAL_HAS_MATCH,AVAILABILITY_AGREEMENT
0,R0002,2,3,32,4,1,31,3,False,True,True,True
1,R0003,5,6,19,0,0,19,0,False,True,False,False
2,R0009,3,4,4,3,3,1,0,False,True,True,True
3,R0015,5,5,500,32,32,468,0,False,True,True,True
4,R0018,3,3,500,188,188,312,0,False,True,True,True
5,R0025,3,3,8,8,8,0,0,True,True,True,True
6,R0030,5,6,0,0,0,0,0,True,False,False,True
7,R0038,3,2,28,46,28,0,18,False,True,True,True
8,R0054,3,3,29,29,29,0,0,True,True,True,True
9,R0062,2,2,25,25,25,0,0,True,True,True,True



STEP 188F: COMPLETED


In [27]:
# STEP 189 — NLP Error Impact Analysis

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Prepare oracle-priority scenario
# --------------------------------------------------

# Keep only NLP 1 predictions found in Ground Truth.
# Their actual priorities are used for diagnostic purposes.

oracle_priority_requirements = (
    batch_nlp1_requirements.merge(
        test_ground_truth[
            [
                "REQUEST_ID",
                "CHARACTERISTIC",
                "VALUE",
                "REQUIREMENT_TYPE"
            ]
        ],
        on=["REQUEST_ID", "CHARACTERISTIC", "VALUE"],
        how="inner",
        validate="one_to_one"
    )
)

# --------------------------------------------------
# 2. Evaluation function
# --------------------------------------------------

def evaluate_matching_scenario(
    predicted_requirements,
    reference_requirements,
    request_ids
):

    records = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for request_id in request_ids:

        predicted = predicted_requirements[
            predicted_requirements["REQUEST_ID"] == request_id
        ]

        actual = reference_requirements[
            reference_requirements["REQUEST_ID"] == request_id
        ]

        predicted_ids = get_eligible_product_ids(predicted)
        actual_ids = get_eligible_product_ids(actual)

        tp = len(predicted_ids & actual_ids)
        fp = len(predicted_ids - actual_ids)
        fn = len(actual_ids - predicted_ids)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        records.append({
            "REQUEST_ID": request_id,
            "PRODUCT_SET_MATCH": predicted_ids == actual_ids,
            "AVAILABILITY_AGREEMENT": (
                bool(predicted_ids) == bool(actual_ids)
            )
        })

    results = pd.DataFrame(records)

    precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp > 0
        else np.nan
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn > 0
        else np.nan
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else np.nan
    )

    return {
        "PRODUCT_SET_EXACT_MATCH": results[
            "PRODUCT_SET_MATCH"
        ].mean(),

        "ELIGIBILITY_PRECISION": precision,

        "ELIGIBILITY_RECALL": recall,

        "ELIGIBILITY_F1": f1,

        "AVAILABILITY_AGREEMENT": results[
            "AVAILABILITY_AGREEMENT"
        ].mean()
    }


# --------------------------------------------------
# 3. Evaluate scenarios
# --------------------------------------------------

request_ids = test_requests["REQUEST_ID"].tolist()

scenario_results = []

for name, requirements in [
    ("A - Ground Truth", test_ground_truth),
    ("B - NLP 1 + Oracle Priority", oracle_priority_requirements),
    ("C - NLP 1 + NLP 2", batch_predicted_requirements)
]:

    metrics = evaluate_matching_scenario(
        requirements,
        test_ground_truth,
        request_ids
    )

    scenario_results.append({
        "SCENARIO": name,
        **metrics
    })

scenario_comparison = pd.DataFrame(scenario_results)

# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

print("=== NLP ERROR IMPACT ANALYSIS ===")

display(
    scenario_comparison.round(4)
)

print("\n=== DIAGNOSTIC DATA ===")

print(
    "NLP 1 predicted requirements:",
    len(batch_nlp1_requirements)
)

print(
    "NLP 1 predictions matching Ground Truth:",
    len(oracle_priority_requirements)
)

print(
    "NLP 1 unmatched predictions excluded from scenario B:",
    len(batch_nlp1_requirements)
    - len(oracle_priority_requirements)
)

print("\nSTEP 189: COMPLETED")

=== NLP ERROR IMPACT ANALYSIS ===


,SCENARIO,PRODUCT_SET_EXACT_MATCH,ELIGIBILITY_PRECISION,ELIGIBILITY_RECALL,ELIGIBILITY_F1,AVAILABILITY_AGREEMENT
0,A - Ground Truth,1.0000,1.0000,1.0000,1.0000,1.0000
1,B - NLP 1 + Oracle Priority,0.8624,0.8786,1.0000,0.9354,0.9899
2,C - NLP 1 + NLP 2,0.3154,0.2758,0.8657,0.4183,0.9564



=== DIAGNOSTIC DATA ===
NLP 1 predicted requirements: 1198
NLP 1 predictions matching Ground Truth: 1128
NLP 1 unmatched predictions excluded from scenario B: 70

STEP 189: COMPLETED


## STEP 190 — Isolate NLP 2 Priority Classification Impact

Compare product matching results using the same correctly
extracted NLP 1 requirements under two priority assignments:

- Scenario B: Ground Truth priorities (oracle).
- Scenario D: NLP 2 predicted priorities.

This isolates the impact of priority classification errors
without introducing false-positive NLP 1 requirements.

In [28]:
# STEP 190 — Isolate NLP 2 Priority Classification Impact

import pandas as pd

# --------------------------------------------------
# 1. Keep only correctly extracted NLP 1 requirements
# --------------------------------------------------

correct_requirement_keys = oracle_priority_requirements[
    ["REQUEST_ID", "CHARACTERISTIC", "VALUE"]
].drop_duplicates()

# --------------------------------------------------
# 2. Attach NLP 2 predicted priorities
# --------------------------------------------------

nlp2_correct_requirements = (
    batch_predicted_requirements.merge(
        correct_requirement_keys,
        on=["REQUEST_ID", "CHARACTERISTIC", "VALUE"],
        how="inner",
        validate="one_to_one"
    )
)

assert len(nlp2_correct_requirements) == len(
    oracle_priority_requirements
), "Requirement sets are not identical."

# --------------------------------------------------
# 3. Verify identical characteristic-value sets
# --------------------------------------------------

key_columns = [
    "REQUEST_ID",
    "CHARACTERISTIC",
    "VALUE"
]

oracle_keys = set(
    map(
        tuple,
        oracle_priority_requirements[key_columns].to_numpy()
    )
)

predicted_keys = set(
    map(
        tuple,
        nlp2_correct_requirements[key_columns].to_numpy()
    )
)

assert oracle_keys == predicted_keys, (
    "Scenarios B and D use different requirements."
)

# --------------------------------------------------
# 4. Evaluate NLP 2 priority impact
# --------------------------------------------------

scenario_d_metrics = evaluate_matching_scenario(
    nlp2_correct_requirements,
    test_ground_truth,
    test_requests["REQUEST_ID"].tolist()
)

# --------------------------------------------------
# 5. Compare B, D and C
# --------------------------------------------------

comparison_190 = pd.DataFrame([
    {
        "SCENARIO": "B - Correct NLP 1 + Oracle Priority",
        **evaluate_matching_scenario(
            oracle_priority_requirements,
            test_ground_truth,
            test_requests["REQUEST_ID"].tolist()
        )
    },
    {
        "SCENARIO": "D - Correct NLP 1 + NLP 2",
        **scenario_d_metrics
    },
    {
        "SCENARIO": "C - Full NLP 1 + NLP 2",
        **evaluate_matching_scenario(
            batch_predicted_requirements,
            test_ground_truth,
            test_requests["REQUEST_ID"].tolist()
        )
    }
])

print("=== ISOLATED NLP 2 IMPACT ===")

print(
    "Correctly extracted requirements:",
    len(nlp2_correct_requirements)
)

display(comparison_190.round(4))

print("\nSTEP 190: COMPLETED")

=== ISOLATED NLP 2 IMPACT ===
Correctly extracted requirements: 1128


,SCENARIO,PRODUCT_SET_EXACT_MATCH,ELIGIBILITY_PRECISION,ELIGIBILITY_RECALL,ELIGIBILITY_F1,AVAILABILITY_AGREEMENT
0,B - Correct NLP 1 + Oracle Priority,0.8624,0.8786,1.0000,0.9354,0.9899
1,D - Correct NLP 1 + NLP 2,0.3423,0.2791,0.8899,0.4249,0.9698
2,C - Full NLP 1 + NLP 2,0.3154,0.2758,0.8657,0.4183,0.9564



STEP 190: COMPLETED
